# 211 — Extract Regulatory Structure

Reads each PDF defined in `data/layer_2/document_config.yaml`, sends the text to Claude Sonnet, and writes per-document CSVs to `data/layer_2/`.

| Output file | Contents |
|---|---|
| `{regulation_id}_sections.csv` | Document sections |
| `{regulation_id}_requirements.csv` | Obligations / requirements |
| `{regulation_id}_thresholds.csv` | Quantitative thresholds |
| `{regulation_id}_references.csv` | Cross-document references |
| `regulations.csv` | One row per regulation (appended) |

Re-runnable: yes — overwrites existing per-document CSVs each run.

In [1]:
import sys, os, json, time, yaml
import pandas as pd
import anthropic
from pathlib import Path
from dotenv import load_dotenv

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.document.utils     import strip_fences, call_claude_stream_json, serialise_row
from src.document.pdf_utils import extract_pdf_pages, batch_to_text
from src.document.config    import load_document_config

LAYER2_DIR       = project_root / 'data' / 'layer_2'
INTERMEDIATE_DIR = LAYER2_DIR / 'intermediate'
PDF_DIR          = LAYER2_DIR / 'regulatory_documents'
CONFIG_PATH      = LAYER2_DIR / 'document_config.yaml'

INTERMEDIATE_DIR.mkdir(exist_ok=True)

client          = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
MODEL           = 'claude-sonnet-4-6'
MAX_TOKENS      = 64000   # claude-sonnet-4-6 supports up to 64k output tokens; raised to fit verbatim section text
CHAR_LIMIT      = 100_000
PAGES_PER_BATCH = 10

print(f'Project root     : {project_root}')
print(f'Intermediate dir : {INTERMEDIATE_DIR}')
print(f'PDF dir          : {PDF_DIR}')
print(f'Config           : {CONFIG_PATH}')


Project root     : /Users/emilpastor/Documents/github/loanguard-ai
Intermediate dir : /Users/emilpastor/Documents/github/loanguard-ai/data/layer_2/intermediate
PDF dir          : /Users/emilpastor/Documents/github/loanguard-ai/data/layer_2/regulatory_documents
Config           : /Users/emilpastor/Documents/github/loanguard-ai/data/layer_2/document_config.yaml


## Step 1 — Load document config

In [2]:
config = load_document_config(CONFIG_PATH)
docs   = config['documents']

print(f'Documents in config: {len(docs)}')
for d in docs:
    pdf_path = PDF_DIR / d['pdf_path']
    flag = '\u2713' if pdf_path.exists() else '\u2717 MISSING'
    rid  = d['regulation_id']
    name = d['name']
    print(f'  {flag}  {rid} \u2014 {name}')


Documents in config: 3
  ✓  APS-112 — Capital Adequacy: Standardised Approach to Credit Risk
  ✓  APG-223 — Residential Mortgage Lending
  ✓  APS-220 — Credit Risk Management


## Step 2 — PDF text extraction

In [3]:
# extract_pdf_pages() and batch_to_text() are imported from src.document.pdf_utils
print('PDF utilities loaded from src.document.pdf_utils')

PDF utilities loaded from src.document.pdf_utils


## Step 3 — Claude structure extraction

In [4]:
SYSTEM_PROMPT = (
    'You are a regulatory document analyst. Extract the structure from the '
    'provided regulatory document text and return valid JSON.\n\n'
    'The document text contains page markers in the format "--- PAGE N ---". '
    'Use these markers to record which pages each section spans.\n\n'
    'The JSON must have exactly these four top-level keys:\n\n'
    '1. sections -- array of document sections. Each section object has:\n'
    '   section_id: unique ID using the prefix provided in the user message (e.g. PREFIX-S1, PREFIX-ATT-A)\n'
    '   section_number: the number or letter as it appears in the document\n'
    '   title: the section title\n'
    '   content_summary: a paraphrased summary (NOT verbatim text, max 3 sentences)\n'
    '   text: the complete verbatim text of this section exactly as it appears in the document, '
    'including all paragraphs, sub-points and tables — do NOT paraphrase or truncate\n'
    '   section_type: one of core | attachment | table | division | chapter\n'
    '   paragraph_range: e.g. 1-15 or A1-A20 or whole\n'
    '   source_page_start: integer — the page number where this section begins (from the --- PAGE N --- markers)\n'
    '   source_page_end: integer — the page number where this section ends\n\n'
    '2. requirements -- array of obligations. Each requirement object has:\n'
    '   requirement_id: sequential ID using the prefix (e.g. PREFIX-REQ-001)\n'
    '   section_id: must match a section_id from the sections array\n'
    '   description: paraphrased obligation (NOT verbatim, max 2 sentences)\n'
    '   requirement_type: one of serviceability | collateral | capital | origination | monitoring | governance | reporting | disclosure\n'
    '   is_quantitative: true if the requirement contains numeric thresholds, otherwise false\n'
    '   applies_to_loan_type: one of residential_secured | commercial_secured | all\n'
    '   severity: use the severity value provided in the user message\n\n'
    '3. thresholds -- array of quantitative thresholds (only for is_quantitative requirements). Each threshold object has:\n'
    '   threshold_id: sequential ID (e.g. PREFIX-THR-001)\n'
    '   requirement_id: must match a requirement_id from the requirements array\n'
    '   metric: what is measured (e.g. LVR, risk_weight, interest_rate_buffer)\n'
    '   operator: one of <= | >= | == | between\n'
    '   value: the primary numeric value as a number\n'
    '   value_upper: upper bound if operator is between, otherwise null\n'
    '   unit: e.g. percent | AUD | basis_points\n'
    '   condition_context: a JSON object describing when this threshold applies (e.g. loan_type, LVR band, LMI status)\n'
    '   consequence: what happens when this threshold applies or is breached\n'
    '   threshold_type: classify how this threshold works — use exactly one of:\n'
    '     minimum     : a floor the entity must meet or exceed '
    '(e.g. serviceability_buffer >= 3%, haircut >= 20%); compliant when condition is True\n'
    '     maximum     : a ceiling the entity must not exceed '
    '(e.g. LVR <= 80%); breach when condition is False\n'
    '     trigger     : a monitoring threshold that fires a concern when condition is met '
    '(e.g. LVR >= 90% triggers senior management review); requires_review when True\n'
    '     informational: a reference value used in ADI-level calculations, not a '
    'per-entity pass/fail gate (e.g. risk_weight, LMI_loss_coverage, CCF)\n\n'
    '4. references -- cross-document references. Each reference object has:\n'
    '   from_section_id: the section containing the reference\n'
    '   referenced_document: e.g. APS 220 or CPS 231\n'
    '   referenced_section_hint: a text hint about what is referenced\n'
    '   reference_type: one of requires | elaborates | supplements | supersedes\n'
    '   description: what the reference says\n\n'
    'Return ONLY valid JSON. No markdown code fences. No preamble.'
)


def close_page_gaps(extracted: dict, pages_by_num: dict, total_pages: int) -> dict:
    """Extend each section's source_page_end to fill gaps before the next section starts.

    Claude correctly identifies section content boundaries but leaves unclaimed pages
    (e.g. cover page, ToC) between sections.  This post-process:
      1. Extends source_page_end to next_section.source_page_start - 1 where a gap exists.
      2. Appends the raw PDF text from those gap pages to the section's text field so
         notebook 213 chunks all content without data loss or gap-fill fallback.
    The last section's page_end is extended to total_pages if the PDF has trailing pages.
    """
    sections = extracted.get('sections', [])
    if not sections:
        return extracted

    sorted_secs = sorted(sections, key=lambda s: s.get('source_page_start') or 0)

    for i, sec in enumerate(sorted_secs):
        p_end = sec.get('source_page_end')
        if p_end is None:
            continue

        next_start = (
            sorted_secs[i + 1].get('source_page_start')
            if i + 1 < len(sorted_secs)
            else total_pages + 1
        )
        if next_start is None or int(p_end) >= int(next_start) - 1:
            continue

        gap_pages = range(int(p_end) + 1, int(next_start))
        gap_text  = '\n\n'.join(
            pages_by_num[pn].strip()
            for pn in gap_pages
            if pages_by_num.get(pn, '').strip()
        )
        if gap_text:
            sec['text'] = ((sec.get('text') or '') + '\n\n' + gap_text).strip()
        sec['source_page_end'] = int(next_start) - 1
        print(f'    Gap-closed {sec["section_id"]}: source_page_end {p_end} → {int(next_start) - 1} '
              f'(absorbed page(s) {list(gap_pages)})')

    extracted['sections'] = sorted_secs
    return extracted


def call_claude(doc: dict, text: str, batch_note: str = '') -> dict:
    """Extract regulatory structure from document text via Claude."""
    prefix   = doc['section_id_prefix']
    severity = doc['default_severity']
    system   = SYSTEM_PROMPT.replace('PREFIX', prefix)

    user_lines = [
        f'Document prefix : {prefix}',
        f'Default severity: {severity}',
        '',
        doc['supplemental_prompt'],
    ]
    if batch_note:
        user_lines.append(f'NOTE: {batch_note}')
    user_lines += ['', '---', '', 'DOCUMENT TEXT:', text]
    user_msg = '\n'.join(user_lines)

    return call_claude_stream_json(
        client,
        model=MODEL,
        max_tokens=MAX_TOKENS,
        system=system,
        messages=[{'role': 'user', 'content': user_msg}],
    )


def merge_batches(results: list) -> dict:
    """Merge extraction results from multiple page batches."""
    merged = {'sections': [], 'requirements': [], 'thresholds': [], 'references': []}
    for r in results:
        for key in merged:
            merged[key].extend(r.get(key, []))
    return merged


def extract_document(doc: dict) -> dict:
    """Extract structure from a single PDF. Batches pages if document is very large."""
    pdf_path = PDF_DIR / doc['pdf_path']
    print(f'  Reading: {pdf_path.name}')

    pages        = extract_pdf_pages(pdf_path)
    pages_by_num = {pn: text for pn, text in pages}
    total_pages  = max(pages_by_num) if pages_by_num else 0
    full_text    = batch_to_text(pages)
    print(f'  Pages: {total_pages} | Characters: {len(full_text):,}')

    if len(full_text) <= CHAR_LIMIT:
        print(f'  Sending full text in single API call...')
        t0      = time.perf_counter()
        result  = call_claude(doc, full_text)
        elapsed = time.perf_counter() - t0
        print(f'  API call completed in {elapsed:.1f}s')
        return close_page_gaps(result, pages_by_num, total_pages)

    batches = [pages[i:i + PAGES_PER_BATCH] for i in range(0, len(pages), PAGES_PER_BATCH)]
    print(f'  Splitting into {len(batches)} batches of {PAGES_PER_BATCH} pages...')

    results     = []
    batch_times = []
    for i, batch in enumerate(batches):
        page_nums = [p[0] for p in batch]
        note = (
            f'Batch {i + 1} of {len(batches)}, covering pages {page_nums[0]}-{page_nums[-1]} '
            f'of {total_pages} total. Continue sequential IDs from previous batches.'
        )
        print(f'  Batch {i + 1}/{len(batches)}: pages {page_nums[0]}-{page_nums[-1]}...', end=' ', flush=True)
        t0 = time.perf_counter()
        results.append(call_claude(doc, batch_to_text(batch), batch_note=note))
        elapsed = time.perf_counter() - t0
        batch_times.append(elapsed)
        print(f'{elapsed:.1f}s')
        if i < len(batches) - 1:
            time.sleep(1)

    total_api_time = sum(batch_times)
    print(f'  Batches done — total API time: {total_api_time:.1f}s '
          f'(avg {total_api_time / len(batch_times):.1f}s/batch)')
    merged = merge_batches(results)
    return close_page_gaps(merged, pages_by_num, total_pages)


## Step 4 — Process all documents

In [5]:
# serialise_row() is imported from src.document.utils

def write_csvs(doc: dict, extracted: dict):
    rid = doc['regulation_id']
    for key, suffix in [
        ('sections',     'sections'),
        ('requirements', 'requirements'),
        ('thresholds',   'thresholds'),
        ('references',   'references'),
    ]:
        rows = [serialise_row(r) for r in extracted.get(key, [])]
        df = pd.DataFrame(rows)
        if not df.empty:
            df.insert(0, 'regulation_id', rid)
        df.to_csv(INTERMEDIATE_DIR / f'{rid}_{suffix}.csv', index=False)
        print(f'  {suffix.capitalize():<15}: {len(df)} rows')


def append_regulation_row(doc: dict):
    reg_path = LAYER2_DIR / 'regulations.csv'
    row = pd.DataFrame([{
        'regulation_id':  doc['regulation_id'],
        'name':           doc['name'],
        'issuing_body':   doc['issuing_body'],
        'document_type':  doc['document_type'],
        'effective_date': doc['effective_date'],
        'is_enforceable': doc['is_enforceable'],
    }])
    if reg_path.exists():
        row.to_csv(reg_path, mode='a', header=False, index=False)
    else:
        row.to_csv(reg_path, index=False)


# Clear regulations.csv for a clean run
reg_csv = LAYER2_DIR / 'regulations.csv'
if reg_csv.exists():
    reg_csv.unlink()
    print('Cleared existing regulations.csv')

all_extracted  = {}
doc_times      = {}
pipeline_start = time.perf_counter()

for doc in docs:
    rid = doc['regulation_id']
    print(f'\n{"=" * 60}')
    print(f'Processing: {rid}')
    print('=' * 60)
    doc_start = time.perf_counter()
    extracted = extract_document(doc)
    write_csvs(doc, extracted)
    append_regulation_row(doc)
    elapsed = time.perf_counter() - doc_start
    doc_times[rid] = elapsed
    all_extracted[rid] = extracted
    print(f'  \u2713 Done  ({elapsed:.1f}s)')

pipeline_elapsed = time.perf_counter() - pipeline_start
print(f'\nAll {len(docs)} documents processed successfully.')
print(f'\nProcessing time summary:')
for rid, t in doc_times.items():
    print(f'  {rid:<12} {t:.1f}s')
print(f'  {"TOTAL":<12} {pipeline_elapsed:.1f}s')



Processing: APS-112
  Reading: APS_112_Capital_Adequacy.pdf
  Pages: 59 | Characters: 120,100
  Splitting into 6 batches of 10 pages...
  Batch 1/6: pages 1-10... 136.1s
  Batch 2/6: pages 11-20... 305.0s
  Batch 3/6: pages 21-30... 252.5s
  Batch 4/6: pages 31-40... 155.9s
  Batch 5/6: pages 41-50... 201.5s
  Batch 6/6: pages 51-59... 100.8s
  Batches done — total API time: 1151.9s (avg 192.0s/batch)
  Sections       : 62 rows
  Requirements   : 135 rows
  Thresholds     : 213 rows
  References     : 45 rows
  ✓ Done  (1160.7s)

Processing: APG-223
  Reading: APG_223_Residential_Mortgage_Lending.pdf
  Pages: 27 | Characters: 71,989
  Sending full text in single API call...
  API call completed in 577.7s
  Sections       : 10 rows
  Requirements   : 72 rows
  Thresholds     : 5 rows
  References     : 14 rows
  ✓ Done  (579.7s)

Processing: APS-220
  Reading: APS_220_Credit_Risk_Management.pdf
  Pages: 30 | Characters: 71,378
  Sending full text in single API call...
  API call comple

## Step 5 — Validation and summary

In [6]:
print('\n' + '=' * 60)
print('VALIDATION SUMMARY')
print('=' * 60)

total_errors = 0

for rid, extracted in all_extracted.items():
    sections     = extracted.get('sections', [])
    requirements = extracted.get('requirements', [])
    thresholds   = extracted.get('thresholds', [])
    references   = extracted.get('references', [])

    section_ids = {s.get('section_id') for s in sections}
    req_ids     = {r.get('requirement_id') for r in requirements}

    orphan_reqs = [r for r in requirements if r.get('section_id') not in section_ids]
    orphan_thr  = [t for t in thresholds  if t.get('requirement_id') not in req_ids]
    errors = len(orphan_reqs) + len(orphan_thr)
    total_errors += errors

    status = '\u2713' if errors == 0 else f'\u26a0 {errors} errors'
    print(f'\n  {rid} [{status}]')
    print(f'    Sections:     {len(sections)}')
    print(f'    Requirements: {len(requirements)}')
    print(f'    Thresholds:   {len(thresholds)}')
    print(f'    References:   {len(references)}')
    if orphan_reqs:
        print(f'    \u26a0  {len(orphan_reqs)} requirement(s) with unknown section_id')
    if orphan_thr:
        print(f'    \u26a0  {len(orphan_thr)} threshold(s) with unknown requirement_id')

print(f'\nTotal validation errors: {total_errors}')
print('regulations.csv written \u2713')


VALIDATION SUMMARY

  APS-112 [✓]
    Sections:     62
    Requirements: 135
    Thresholds:   213
    References:   45

  APG-223 [✓]
    Sections:     10
    Requirements: 72
    Thresholds:   5
    References:   14

  APS-220 [✓]
    Sections:     26
    Requirements: 57
    Thresholds:   26
    References:   14

Total validation errors: 0
regulations.csv written ✓
